<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/_banners/opim5641_banner.svg" width="100%" alt="OPIM 5641 banner"/>

**Dr. Dave Wanik - Operations and Information Management - University of Connecticut**

---

# Studio 4.1 — Regression as an Optimization Problem

A regression model is an optimization problem at heart: choose the parameters (like $a$ and $b$) that make the
distance between the actual and predicted values as small as possible.

When you write that optimization model yourself, **you** make two choices that software usually makes for you:

1. **The form of the model**: a straight line, a power curve, a polynomial...
2. **The error you minimize**: squared error, absolute error...

Change either one and you get a **different answer**, and some curves will simply look better to the eye than
others.

### How we secretly code a regression in Pyomo

The trick from the pharmacy lecture, in one picture. We make a decision variable $y_i$ for every store's
prediction, and on its own each $y_i$ could be **any** number. Then one constraint per store says it can't be just
anything: it must follow the form of the model, $y_i = a + b\,x_i$. Only then does the objective compare each
$y_i$ with the actual revenue $z_i$.

<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/figures/studio4_1_placeholder_trick.png" width="100%"/>

Written as one optimization model:

$$
\begin{aligned}
\min_{a,\,b,\,y_0,\dots,y_9} \quad & \sum_{i=0}^{9} (y_i - z_i)^2 && \text{(objective: the error)} \\
\text{subject to} \quad & y_i = a + b\,x_i \qquad i = 0, 1, \dots, 9 && \text{(constraints: the form of the model)} \\
& a,\ b,\ y_i \in \mathbb{R} && \text{(decision variables)}
\end{aligned}
$$

### Where you are headed

Tonight you build the model once, then play: every scenario you run is added to one picture so you can compare
them side by side.

<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/figures/studio4_1_all_scenarios.png" width="90%"/>

## Import modules (download Pyomo and the ipopt solver)

Run this first, every time. It installs Pyomo and **ipopt**, the nonlinear solver we use in Module 4.
It takes about a minute.

In [ ]:
# install Pyomo and the ipopt solver in Colab (idaes-pse brings both)
import sys
import os

if 'google.colab' in sys.modules:
    !pip install -q idaes-pse --pre
    !idaes get-extensions --to ./bin
    os.environ['PATH'] += ':bin'

# this makes plots using pylab (matplotlib)
%matplotlib inline
from pylab import *
import numpy as np
import pandas as pd

# import ALL NAMESPACES from pyomo
from pyomo.environ import *

# where the setup cell put the solver
ipopt_path = '/content/bin/ipopt'

# Part 1 · The pharmacy problem

A pharmacy has hired you to predict weekly **revenue** from the number of **hours** the store is open. Here are
ten stores (168 hours is a store open 24/7):

| Hours | 40 | 44 | 48 | 48 | 60 | 70 | 72 | 90 | 100 | 168 |
|---|---|---|---|---|---|---|---|---|---|---|
| Revenue (\$) | 5,958 | 6,662 | 6,004 | 6,011 | 7,250 | 8,632 | 6,964 | 11,097 | 9,107 | 11,498 |

As in M4.1, we keep the data separate from the model: `X` holds the hours, `Z` holds the actual revenue.

In [ ]:
# read the data
pharmacy_df = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/7_Nonlinear/data/pharmacy.csv')

# X = hours (the input), Z = revenue (the actual value we want to predict)
X = []
Z = []
for i in range(len(pharmacy_df)):
    X.append(float(pharmacy_df.loc[i, 'hours']))
    Z.append(float(pharmacy_df.loc[i, 'revenue']))

# this is how many observations (stores) we have
n_points = len(X)

print("X (hours):  ", X)
print("Z (revenue):", Z)
print("n_points =", n_points)

In [ ]:
# look before you model
scatter(X, Z, color='black')
xlabel('Hours open per week')
ylabel('Weekly revenue ($)')
title('Revenue vs. hours: ten pharmacies')
show()

# Part 2 · The model on paper

### The decision variables

- The **parameters** of the model: $a$ and $b$ (and $c$ if the model needs a third one). These are what the solver chooses.
- One **placeholder prediction per store**: $y_0, y_1, \dots, y_9$. This is the big leap. On its own, each $y_i$
  could take on **any** value. Ten stores, ten placeholders.

### The constraints: where the form of the model lives

Here is the sneaky part: the regression formula is **not** in the objective. It is in the **constraints**, one per
store. Each constraint forces that store's placeholder to follow the model:

| Model form | One constraint per store $i$ | Parameters |
|---|---|---|
| Linear | $y_i = a + b\,x_i$ | $a, b$ |
| Power | $y_i = a \cdot x_i^{\,b}$ | $a, b$ |
| Quadratic (a polynomial) | $y_i = a + b\,x_i + c\,x_i^2$ | $a, b, c$ |
| The crazy one (seven roots) | $y_i = a_1 x_i + a_2 x_i^{1/2} + a_3 x_i^{1/3} + \dots + a_7 x_i^{1/7} = \displaystyle\sum_{k=1}^{7} a_k\, x_i^{1/k}$ | $a_1, \dots, a_7$ |

Notice that $a$ and $b$ are the **same** in every constraint: all ten stores share one model. Written out for the
linear model, the ten constraints are:

$$
\begin{aligned}
y_0 &= a + b \cdot 40 \\
y_1 &= a + b \cdot 44 \\
&\ \ \vdots \\
y_9 &= a + b \cdot 168
\end{aligned}
$$

Ten equations, but only two numbers ($a$ and $b$) to choose. That is why the placeholders cannot each go wherever
they like.

### The objective: the error you minimize

Now the objective only has to compare each placeholder $y_i$ with the actual value $z_i$:

| Error | Objective |
|---|---|
| Squared error (the usual one) | $\min \sum_{i} (y_i - z_i)^2$ |
| Absolute error | $\min \sum_{i} \lvert y_i - z_i \rvert$ |

Squared error punishes one big miss much more than several small ones: a miss of \$1,000 costs
$1000^2 = 1{,}000{,}000$, while two misses of \$500 cost only $2 \times 500^2 = 500{,}000$. Absolute error treats
every dollar of miss the same (\$1,000 either way).

### Putting it together

Every scenario tonight is the same three pieces. Only the constraint (the form) or the objective (the error) changes:

| | Objective (the error) | Constraint, one per store (the form) |
|---|---|---|
| Linear, squared | $\min \sum_i (y_i - z_i)^2$ | $y_i = a + b\,x_i$ |
| Linear, absolute | $\min \sum_i \lvert y_i - z_i \rvert$ | $y_i = a + b\,x_i$ |
| Power, squared | $\min \sum_i (y_i - z_i)^2$ | $y_i = a \cdot x_i^{\,b}$ |
| Quadratic, squared | $\min \sum_i (y_i - z_i)^2$ | $y_i = a + b\,x_i + c\,x_i^2$ |
| Linear, never over-predict | $\min \sum_i (y_i - z_i)^2$ | $y_i = a + b\,x_i$ **and** $y_i \le z_i$ |
| The crazy one, absolute | $\min \sum_i \lvert y_i - z_i \rvert$ | $y_i = \sum_{k=1}^{7} a_k\, x_i^{1/k}$ |

**The answer to check against.** `sklearn` (scikit-learn, the machine learning library you may know from
other classes) fits the ordinary least-squares line in a few lines of code. Our optimization model (linear form,
squared error) must give exactly the same $a$ and $b$. If it does, the model is right.

In [ ]:
from sklearn.linear_model import LinearRegression

# sklearn wants X as a column (10 rows, 1 column)
X_column = np.array(X).reshape(-1, 1)

ols = LinearRegression()
ols.fit(X_column, Z)
print("sklearn:  a = %.2f   b = %.4f" % (ols.intercept_, ols.coef_[0]))

**The scoreboard.** Empty dictionaries. Every scenario you run adds one entry with what the model found
(`scoreboard[scenario] = ...`), its fitted equation and its curve, and Part 5 shows all of them together. Run this
cell **once**: running it again wipes the scoreboard clean.

In [ ]:
# scenario name -> what the model found
scoreboard = {}
# scenario name -> the fitted equation (LaTeX)
equations = {}
# scenario name -> the fitted curve, drawn over these hours
curves = {}
grid = np.linspace(30, 175, 300)

# Part 3 · The scenario

Leave it alone for the first run (the base scenario: linear model, squared error). Then try the others **one at a
time**: remove the `#` from that scenario's lines, click on the cell, and choose **Runtime → Run cell and below**
(Ctrl+F10). Put the `#` back when you are done.

In [ ]:
# ---------- THE SCENARIO (change things here) ----------
scenario = "1 linear, squared error"
model_form = "linear"        # "linear", "power", "quadratic" or "crazy"
error_type = "squared"       # "squared" or "absolute"
never_overpredict = False    # a business rule (Scenario 6)

# Scenario 2: same line, absolute error
# scenario = "2 linear, absolute error"
# error_type = "absolute"

# Scenario 3: a power model
# scenario = "3 power, squared error"
# model_form = "power"

# Scenario 4: a power model with absolute error
# scenario = "4 power, absolute error"
# model_form = "power"
# error_type = "absolute"

# Scenario 5: a polynomial (quadratic) model
# scenario = "5 quadratic, squared error"
# model_form = "quadratic"

# Scenario 6: finance wants a budget-safe line that never predicts more than a store actually made
# scenario = "6 linear, never over-predict"
# never_overpredict = True

# Scenario 7: the crazy one from the pharmacy lecture - seven terms, one for each root of x
# scenario = "7 crazy seven-term, absolute error"
# model_form = "crazy"
# error_type = "absolute"

# Scenario 8: your own idea (mix and match!)


# Part 4 · Build it, one piece at a time

**4a · The decision variables.** The parameters $a$ and $b$ (plus $c$ for the quadratic, or $a_1, \dots, a_7$ for
the crazy one), and the ten placeholder predictions `model.y[i]`:

$$a,\ b \in \mathbb{R} \qquad\qquad y_0,\ y_1,\ \dots,\ y_9 \in \mathbb{R}$$

Nonlinear solvers need a sensible starting point (the hill climber in the fog!), so we initialize everything.

Look at the printout: each `y` has **no bounds and no rule** yet. Right now it could be anything.

In [ ]:
# ---------- 4a. the decision variables ----------
model = ConcreteModel()

# the parameters of the model
model.a = Var(domain=Reals, initialize=1000)
model.b = Var(domain=Reals, initialize=1)
if model_form == "power":
    model.b = 0.5        # a power model starts near a square root
if model_form == "quadratic":
    model.c = Var(domain=Reals, initialize=0)
if model_form == "crazy":
    model.ak = Var(range(1, 8), domain=Reals, initialize=1)    # a_1 ... a_7, one per root

# one placeholder prediction per store - on its own, each one can be anything
model.y = Var(range(n_points), domain=Reals)
for i in range(n_points):
    model.y[i] = 0.0

model.y.pprint()

**4b · The constraints: knob 1, the form of the model.** One constraint per store. This is where each
placeholder `y[i]` gets **forced** to follow the model:

$$y_i = f(x_i) \qquad \text{for every store } i = 0, 1, \dots, 9$$

where $f$ is the form you picked ($a + b\,x$, $a\,x^b$, $a + b\,x + c\,x^2$ or $\sum_k a_k\,x^{1/k}$). Look at what
Pyomo prints: the prediction minus the model has to equal 0, and the same parameters appear in every row.

In [ ]:
# ---------- 4b. one constraint per store: the form of the model ----------
model.constraints = ConstraintList()

for i in range(n_points):
    x = X[i]
    if model_form == "linear":
        model.constraints.add(model.y[i] == model.a + model.b * x)
    if model_form == "power":
        model.constraints.add(model.y[i] == model.a * x ** model.b)
    if model_form == "quadratic":
        model.constraints.add(model.y[i] == model.a + model.b * x + model.c * x ** 2)
    if model_form == "crazy":
        # a_1 * x + a_2 * x**(1/2) + ... + a_7 * x**(1/7)
        seven_terms = 0
        for k in range(1, 8):
            seven_terms += model.ak[k] * x ** (1 / k)
        model.constraints.add(model.y[i] == seven_terms)

# show the first three constraints (there are n_points of them)
print(model_form, "model:", len(model.constraints), "constraints, one per store. The first three:")
for k in [1, 2, 3]:
    print("  ", model.constraints[k].expr)

**4c · The objective: knob 2, the error.** The `+=` loop adds one store's error at a time.

For **absolute error** we cannot use `abs()`: the solver follows the slope of the objective downhill, and a V has
a sharp point with no slope at the bottom, exactly where we want to end up. So we make one more variable per store,
`model.d[i]`, and give it **two floors**: `d >= miss` and `d >= -miss`.

| the store's miss ($y_i - z_i$) | floor 1: d >= miss | floor 2: d >= -miss | smallest d allowed |
|---|---|---|---|
| +300 (we over-predicted) | d >= 300 | d >= -300 | **300** |
| -200 (we under-predicted) | d >= -200 | d >= 200 | **200** |

One of the two floors is always the positive size of the miss. Because we **minimize** the total of the `d`'s, the
solver pushes every `d` down until it hits the higher floor, which is exactly $\lvert y_i - z_i \rvert$. As math:

$$
\min \sum_{i} \lvert y_i - z_i \rvert
\quad \Longleftrightarrow \quad
\min \sum_{i} d_i \;\; \text{ subject to } \;\; d_i \ge y_i - z_i, \;\; d_i \ge -(y_i - z_i), \;\; d_i \ge 0
$$

In [ ]:
# ---------- 4c. the objective: the error you minimize ----------
obj_expr = 0

if error_type == "squared":
    for i in range(n_points):
        obj_expr += (model.y[i] - Z[i]) ** 2

if error_type == "absolute":
    # d[i] = |y[i] - Z[i]| at the optimum (two floors per store)
    model.d = Var(range(n_points), domain=NonNegativeReals)
    for i in range(n_points):
        model.constraints.add(model.d[i] >= model.y[i] - Z[i])
        model.constraints.add(model.d[i] >= -(model.y[i] - Z[i]))
        obj_expr += model.d[i]

model.error = Objective(expr=obj_expr, sense=minimize)
print("minimize", error_type, "error:")
print(obj_expr)

**4d · A business rule (Scenario 6).** Plain regression has no way to say "never predict more than a store
actually made". In an optimization model it is one more constraint per store:

$$y_i \le z_i \qquad \text{for every store } i$$

(Adding a constraint can never make the error *smaller*; watch what it costs.)

In [ ]:
# ---------- 4d. a business rule ----------
if never_overpredict:
    for i in range(n_points):
        model.constraints.add(model.y[i] <= Z[i])
    print("rule: never over-predict (", n_points, "more constraints )")
else:
    print("no business rule in this scenario")

**4e · Solve it and read the results.** We show the fitted model as an equation, the total squared error
(one yardstick for every scenario),

$$\text{SSE} = \sum_{i=0}^{9} (y_i - z_i)^2,$$

and a prediction for a store open 120 hours, a store we have no data for. (If the solver prints a warning, read it:
"maximum number of iterations" means ipopt gave up before it found the bottom of the hill.)

In [ ]:
# ---------- 4e. solve and report ----------
from IPython.display import display, Math

results = SolverFactory('ipopt', executable=ipopt_path).solve(model)
print("SCENARIO:", scenario)
print("Termination:", results.solver.termination_condition)

a = model.a()
b = model.b()
c = 0
if model_form == "quadratic":
    c = model.c()
ak = {}
if model_form == "crazy":
    a = np.nan
    b = np.nan
    for k in range(1, 8):
        ak[k] = model.ak[k]()

# the fitted model as an equation
if model_form == "linear":
    fitted = r"\hat{y} = %.2f + %.4f\,x" % (a, b)
if model_form == "power":
    fitted = r"\hat{y} = %.2f\,x^{%.4f}" % (a, b)
if model_form == "quadratic":
    fitted = r"\hat{y} = %.2f + %.4f\,x %+.6f\,x^2" % (a, b, c)
if model_form == "crazy":
    fitted = r"\hat{y} ="
    for k in range(1, 8):
        power = "x^{1/%d}" % k
        if k == 1:
            power = "x"
        number = "{:+,.0f}".format(ak[k]).replace(",", "{,}")     # {,} keeps LaTeX from spacing the commas
        fitted += " " + number + r"\," + power
display(Math(fitted))

# the fitted model's prediction for any number of hours h
def predict(h):
    if model_form == "linear":
        return a + b * h
    if model_form == "power":
        return a * h ** b
    if model_form == "quadratic":
        return a + b * h + c * h ** 2
    if model_form == "crazy":
        total = 0
        for k in range(1, 8):
            total += ak[k] * h ** (1 / k)
        return total

# the same yardstick for every scenario
sse = 0
for i in range(n_points):
    sse += (model.y[i]() - Z[i]) ** 2

pred_120 = predict(120)
print("total squared error = {:,.0f}".format(sse))
print("predicted revenue for a store open 120 hours = ${:,.0f}".format(pred_120))

# add this scenario to the scoreboard: scoreboard[key] = value
scoreboard[scenario] = {'form': model_form, 'error': error_type, 'a': a, 'b': b, 'c': c,
                        'squared error': sse, 'revenue at 120 hours': pred_120}
equations[scenario] = fitted
curve = []
for h in grid:
    curve.append(predict(h))
curves[scenario] = curve

# Part 5 · Every scenario, side by side

The equations, the table and the picture grow every time you run a scenario.

In [ ]:
# every fitted model, as an equation
for name in equations:
    print(name)
    display(Math(equations[name]))

In [ ]:
# the scoreboard as a table
pd.DataFrame(scoreboard).T

In [ ]:
# every scenario's curve on one picture
figure(figsize=(10, 6))
scatter(X, Z, color='black', s=50, zorder=3, label='the ten stores')
for name in curves:
    plot(grid, curves[name], linewidth=2, label=name)
ylim(0, 15000)     # keeps the picture readable (a wild curve can leave the frame)
xlabel('Hours open per week')
ylabel('Weekly revenue ($)')
title('Same data, different choices, different answers')
legend(fontsize=8)
show()

### Your turn

Run scenarios 1 to 7, then answer in a text cell:

1. Does Scenario 1 match `sklearn` exactly? Why does that matter?
2. Linear model, squared vs. absolute error (Scenarios 1 and 2): how did $a$ and $b$ change, and why?
3. Which curve looks best to your eye? Which one has the lowest squared error? Are they the same one?
4. The power model's $b$ is below 1. What does that say about the value of staying open longer?
5. Scenario 6 makes the squared error worse. Why is that guaranteed, and why would finance still want it?
6. Scenario 7 (the crazy one) beats the quadratic on squared error. Look at its coefficients, its warning from
   ipopt, and its prediction for 120 hours. Would you hand this model to your boss?

# Appendix · Each model, written cleanly

The notebook above uses switches (`model_form`, `error_type`) so one set of cells can run every scenario. In the
wild you would usually write just the one model you need. Here is each one on its own: the same pieces, nothing
else. Each cell runs by itself after Part 1 (it only needs `X`, `Z` and `n_points`).

## A1 · Linear model, squared error (ordinary least squares)

$$y_i = a + b\,x_i \qquad \min \sum_i (y_i - z_i)^2$$

In [ ]:
m = ConcreteModel()
m.a = Var(initialize=1000)
m.b = Var(initialize=1)
m.y = Var(range(n_points))

m.fit = ConstraintList()
for i in range(n_points):
    m.fit.add(m.y[i] == m.a + m.b * X[i])

error = 0
for i in range(n_points):
    error += (m.y[i] - Z[i]) ** 2
m.obj = Objective(expr=error, sense=minimize)

SolverFactory('ipopt', executable=ipopt_path).solve(m)
print("revenue = %.2f + %.4f * hours" % (m.a(), m.b()))

## A2 · Linear model, absolute error

$$y_i = a + b\,x_i \qquad \min \sum_i d_i \;\text{ with }\; d_i \ge y_i - z_i,\; d_i \ge -(y_i - z_i)$$

In [ ]:
m = ConcreteModel()
m.a = Var(initialize=1000)
m.b = Var(initialize=1)
m.y = Var(range(n_points))
m.d = Var(range(n_points), domain=NonNegativeReals)     # d[i] = |y[i] - Z[i]|

m.fit = ConstraintList()
for i in range(n_points):
    m.fit.add(m.y[i] == m.a + m.b * X[i])
    m.fit.add(m.d[i] >= m.y[i] - Z[i])
    m.fit.add(m.d[i] >= -(m.y[i] - Z[i]))

error = 0
for i in range(n_points):
    error += m.d[i]
m.obj = Objective(expr=error, sense=minimize)

SolverFactory('ipopt', executable=ipopt_path).solve(m)
print("revenue = %.2f + %.4f * hours" % (m.a(), m.b()))

## A3 · Power model, squared error

$$y_i = a \cdot x_i^{\,b} \qquad \min \sum_i (y_i - z_i)^2$$

In [ ]:
m = ConcreteModel()
m.a = Var(initialize=1000)
m.b = Var(initialize=0.5)       # start near a square root
m.y = Var(range(n_points))

m.fit = ConstraintList()
for i in range(n_points):
    m.fit.add(m.y[i] == m.a * X[i] ** m.b)

error = 0
for i in range(n_points):
    error += (m.y[i] - Z[i]) ** 2
m.obj = Objective(expr=error, sense=minimize)

SolverFactory('ipopt', executable=ipopt_path).solve(m)
print("revenue = %.2f * hours ** %.4f" % (m.a(), m.b()))

## A4 · Power model, absolute error

$$y_i = a \cdot x_i^{\,b} \qquad \min \sum_i d_i \;\text{ with }\; d_i \ge y_i - z_i,\; d_i \ge -(y_i - z_i)$$

In [ ]:
m = ConcreteModel()
m.a = Var(initialize=1000)
m.b = Var(initialize=0.5)
m.y = Var(range(n_points))
m.d = Var(range(n_points), domain=NonNegativeReals)

m.fit = ConstraintList()
for i in range(n_points):
    m.fit.add(m.y[i] == m.a * X[i] ** m.b)
    m.fit.add(m.d[i] >= m.y[i] - Z[i])
    m.fit.add(m.d[i] >= -(m.y[i] - Z[i]))

error = 0
for i in range(n_points):
    error += m.d[i]
m.obj = Objective(expr=error, sense=minimize)

SolverFactory('ipopt', executable=ipopt_path).solve(m)
print("revenue = %.2f * hours ** %.4f" % (m.a(), m.b()))

## A5 · Quadratic (polynomial) model, squared error

$$y_i = a + b\,x_i + c\,x_i^2 \qquad \min \sum_i (y_i - z_i)^2$$

In [ ]:
m = ConcreteModel()
m.a = Var(initialize=1000)
m.b = Var(initialize=1)
m.c = Var(initialize=0)
m.y = Var(range(n_points))

m.fit = ConstraintList()
for i in range(n_points):
    m.fit.add(m.y[i] == m.a + m.b * X[i] + m.c * X[i] ** 2)

error = 0
for i in range(n_points):
    error += (m.y[i] - Z[i]) ** 2
m.obj = Objective(expr=error, sense=minimize)

SolverFactory('ipopt', executable=ipopt_path).solve(m)
print("revenue = %.2f + %.4f * hours + %.6f * hours**2" % (m.a(), m.b(), m.c()))

## A6 · Linear model, squared error, never over-predict

$$y_i = a + b\,x_i \;\text{ and }\; y_i \le z_i \qquad \min \sum_i (y_i - z_i)^2$$

In [ ]:
m = ConcreteModel()
m.a = Var(initialize=1000)
m.b = Var(initialize=1)
m.y = Var(range(n_points))

m.fit = ConstraintList()
for i in range(n_points):
    m.fit.add(m.y[i] == m.a + m.b * X[i])
    m.fit.add(m.y[i] <= Z[i])        # the business rule: never predict more than the store made

error = 0
for i in range(n_points):
    error += (m.y[i] - Z[i]) ** 2
m.obj = Objective(expr=error, sense=minimize)

SolverFactory('ipopt', executable=ipopt_path).solve(m)
print("revenue = %.2f + %.4f * hours" % (m.a(), m.b()))

## A7 · The crazy seven-term model, absolute error (as in the pharmacy lecture)

$$y_i = a_1 x_i + a_2 x_i^{1/2} + a_3 x_i^{1/3} + a_4 x_i^{1/4} + a_5 x_i^{1/5} + a_6 x_i^{1/6} + a_7 x_i^{1/7} \qquad \min \sum_i d_i$$

In [ ]:
m = ConcreteModel()
m.a1 = Var(initialize=1)
m.a2 = Var(initialize=1)
m.a3 = Var(initialize=1)
m.a4 = Var(initialize=1)
m.a5 = Var(initialize=1)
m.a6 = Var(initialize=1)
m.a7 = Var(initialize=1)
m.y = Var(range(n_points))
m.d = Var(range(n_points), domain=NonNegativeReals)

m.fit = ConstraintList()
for i in range(n_points):
    x = X[i]
    m.fit.add(m.y[i] == m.a1 * x + m.a2 * x ** (1/2) + m.a3 * x ** (1/3) + m.a4 * x ** (1/4)
                        + m.a5 * x ** (1/5) + m.a6 * x ** (1/6) + m.a7 * x ** (1/7))
    m.fit.add(m.d[i] >= m.y[i] - Z[i])
    m.fit.add(m.d[i] >= -(m.y[i] - Z[i]))

error = 0
for i in range(n_points):
    error += m.d[i]
m.obj = Objective(expr=error, sense=minimize)

results = SolverFactory('ipopt', executable=ipopt_path).solve(m)
print("Termination:", results.solver.termination_condition)    # spoiler: ipopt gives up
print("a1 =", m.a1(), "  a4 =", m.a4(), "  a7 =", m.a7())

## Bottom line

A regression is an optimization model:

- the **decision variables** are the parameters ($a$, $b$, ...) plus one placeholder prediction per row,
- the **constraints** force each placeholder to follow the **form** of the model,
- the **objective** is the **error** you choose to minimize,
- and anything else the business knows can be added as **one more constraint**.

Change the form or the error and you get a different answer. Part of the job is choosing the one that makes sense,
and the lowest error on the data you have is not the same thing as the most sensible model.